# 13 - Silver Customers SCD2

Rebuilds deterministic customer timelines from append-only Bronze, handles late records, and synchronizes SCD2 history with `MERGE INTO`.

In [ ]:
%run ./05_bronze_common

In [ ]:
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window


def run_version(run_id: str) -> int:
    return int(run_id.replace("-", "")[:15], 16)


def merge_quarantine(table_name: str, errors) -> int:
    count = errors.count()
    if count:
        (
            DeltaTable.forName(spark, table_name)
            .alias("target")
            .merge(errors.alias("source"), "target.error_id = source.error_id")
            .whenNotMatchedInsertAll()
            .execute()
        )
    return count


def silver_dq(
    catalog: str,
    source_name: str,
    pipeline_run_id: str,
    valid_count: int,
    errors,
    duplicate_count: int = 0,
) -> None:
    error_counts = errors.groupBy("error_code").count().collect()
    if duplicate_count:
        error_counts.append(Row(error_code="duplicate_key", count=duplicate_count))
    now = datetime.now(timezone.utc)
    rows = [
        Row(
            quality_date=now.date(),
            source_name=source_name,
            pipeline_stage="silver",
            quality_status="valid",
            error_code=None,
            record_count=int(valid_count),
            pipeline_run_id=pipeline_run_id,
            recorded_ts=now,
        )
    ]
    rows.extend(
        Row(
            quality_date=now.date(),
            source_name=source_name,
            pipeline_stage="silver",
            quality_status="error",
            error_code=row.error_code,
            record_count=int(row["count"]),
            pipeline_run_id=pipeline_run_id,
            recorded_ts=now,
        )
        for row in error_counts
    )
    append_delta(
        spark.createDataFrame(rows, DQ_SCHEMA),
        f"{catalog}.control.data_quality_results",
        f"{source_name}_silver_dq",
        run_version(pipeline_run_id),
    )


def silver_error(
    error_code,
    error_reason,
    record_key,
    raw_payload,
    rescued_data,
    source_file,
    batch_id,
    load_date,
    ingest_ts,
    pipeline_run_id,
):
    return [
        F.sha2(
            F.concat_ws(
                "||",
                source_file,
                F.coalesce(record_key, F.lit("")),
                raw_payload,
                error_code,
            ),
            256,
        ).alias("error_id"),
        F.lit("silver").alias("error_stage"),
        error_code.alias("error_code"),
        error_reason.alias("error_reason"),
        record_key.alias("record_key"),
        raw_payload.alias("raw_payload"),
        rescued_data.alias("rescued_data"),
        source_file.alias("source_file"),
        batch_id.alias("batch_id"),
        load_date.alias("load_date"),
        ingest_ts.alias("ingest_ts"),
        F.current_date().alias("quarantine_date"),
        pipeline_run_id.alias("pipeline_run_id"),
    ]

In [ ]:
dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")
catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
job_run_id = dbutils.widgets.get("job_run_id").strip()
spark.conf.set("spark.sql.session.timeZone", "UTC")

pipeline_name = "silver_customers_scd2"
source_name = "customers"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
bronze_table = f"{catalog}.bronze.bronze_customers"
target_table = f"{catalog}.silver.silver_customers"
quarantine_table = f"{catalog}.quarantine.customers_errors"

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`silver`.`silver_customers` (
        customer_sk STRING NOT NULL,
        user_id STRING NOT NULL,
        email STRING,
        phone STRING NOT NULL,
        country STRING,
        created_at TIMESTAMP,
        updated_at TIMESTAMP NOT NULL,
        attribute_hash STRING NOT NULL,
        effective_from TIMESTAMP NOT NULL,
        effective_to TIMESTAMP,
        is_current BOOLEAN NOT NULL,
        ingest_ts TIMESTAMP NOT NULL,
        source_file STRING NOT NULL,
        batch_id STRING NOT NULL,
        load_date DATE NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
    """
)

upsert_audit(catalog, pipeline_run_id, pipeline_name, "full_timeline_rebuild", start_ts, "RUNNING", [bronze_table], job_run_id)
try:
    bronze = spark.table(bronze_table)
    parsed = (
        bronze
        .withColumn("user_id_parsed", F.get_json_object("raw_payload", "$.user_id"))
        .withColumn("email_parsed", F.get_json_object("raw_payload", "$.email"))
        .withColumn("phone_parsed", F.get_json_object("raw_payload", "$.phone"))
        .withColumn("country_parsed", F.get_json_object("raw_payload", "$.country"))
        .withColumn("created_at_parsed", F.expr("try_to_timestamp(get_json_object(raw_payload, '$.created_at'))"))
        .withColumn("updated_at_parsed", F.expr("try_to_timestamp(get_json_object(raw_payload, '$.updated_at'))"))
        .withColumn(
            "error_code",
            F.when(F.col("user_id_parsed").isNull(), F.lit("missing_key"))
            .when(F.col("updated_at_parsed").isNull(), F.lit("invalid_timestamp"))
            .when(F.col("phone_parsed").isNull() | (F.trim("phone_parsed") == ""), F.lit("schema_mismatch"))
            .when(
                F.col("email_parsed").isNotNull()
                & ((F.trim("email_parsed") == "") | ~F.col("email_parsed").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")),
                F.lit("schema_mismatch"),
            ),
        )
        .withColumn(
            "error_reason",
            F.when(F.col("error_code") == "missing_key", F.lit("user_id is null"))
            .when(F.col("error_code") == "invalid_timestamp", F.lit("updated_at cannot be parsed"))
            .when(F.col("phone_parsed").isNull() | (F.trim("phone_parsed") == ""), F.lit("phone is required"))
            .when(F.col("error_code") == "schema_mismatch", F.lit("email format is invalid")),
        )
    )
    errors = parsed.filter(F.col("error_code").isNotNull()).select(
        *silver_error(
            F.col("error_code"), F.col("error_reason"), F.col("user_id_parsed"),
            F.col("raw_payload"), F.lit(None).cast("string"), F.col("source_file"),
            F.col("batch_id"), F.col("load_date"), F.col("ingest_ts"),
            F.lit(pipeline_run_id),
        )
    )
    error_count = merge_quarantine(quarantine_table, errors)
    valid = parsed.filter(F.col("error_code").isNull()).select(
        F.col("user_id_parsed").alias("user_id"), F.col("email_parsed").alias("email"),
        F.col("phone_parsed").alias("phone"), F.col("country_parsed").alias("country"),
        F.col("created_at_parsed").alias("created_at"), F.col("updated_at_parsed").alias("updated_at"),
        "ingest_ts", "source_file", "batch_id", "load_date",
    ).withColumn(
        "attribute_hash",
        F.sha2(F.concat_ws("||", F.coalesce("email", F.lit("")), F.coalesce("phone", F.lit("")), F.coalesce("country", F.lit(""))), 256),
    )

    exact_rank = Window.partitionBy("user_id", "updated_at").orderBy(F.col("ingest_ts").desc(), F.col("source_file").desc())
    exact = valid.withColumn("_rank", F.row_number().over(exact_rank)).filter(F.col("_rank") == 1).drop("_rank")
    ordered = Window.partitionBy("user_id").orderBy("updated_at")
    changed = exact.withColumn("_previous_hash", F.lag("attribute_hash").over(ordered)).filter(F.col("_previous_hash").isNull() | (F.col("_previous_hash") != F.col("attribute_hash"))).drop("_previous_hash")
    timeline = (
        changed
        .withColumn("effective_from", F.col("updated_at"))
        .withColumn("next_effective_from", F.lead("updated_at").over(ordered))
        .withColumn("effective_to", F.when(F.col("next_effective_from").isNotNull(), F.expr("next_effective_from - INTERVAL 1 MICROSECOND")))
        .withColumn("is_current", F.col("next_effective_from").isNull())
        .withColumn("customer_sk", F.sha2(F.concat_ws("||", "user_id", F.col("effective_from").cast("string")), 256))
        .withColumn("pipeline_run_id", F.lit(pipeline_run_id))
        .drop("next_effective_from")
    )

    target_df = spark.table(target_table)
    stale = target_df.select("customer_sk").join(timeline.select("customer_sk"), "customer_sk", "left_anti").withColumn("_delete", F.lit(True))
    source = timeline.withColumn("_delete", F.lit(False)).unionByName(stale, allowMissingColumns=True)
    (
        DeltaTable.forName(spark, target_table).alias("target")
        .merge(source.alias("source"), "target.customer_sk = source.customer_sk")
        .whenMatchedDelete(condition="source._delete")
        .whenMatchedUpdateAll(condition="NOT source._delete AND NOT (target.email <=> source.email AND target.phone <=> source.phone AND target.country <=> source.country AND target.created_at <=> source.created_at AND target.updated_at <=> source.updated_at AND target.attribute_hash <=> source.attribute_hash AND target.effective_from <=> source.effective_from AND target.effective_to <=> source.effective_to AND target.is_current <=> source.is_current AND target.ingest_ts <=> source.ingest_ts AND target.source_file <=> source.source_file AND target.batch_id <=> source.batch_id AND target.load_date <=> source.load_date)")
        .whenNotMatchedInsertAll(condition="NOT source._delete")
        .execute()
    )
    valid_count = timeline.count()
    silver_dq(catalog, source_name, pipeline_run_id, valid_count, errors)
    upsert_audit(catalog, pipeline_run_id, pipeline_name, "full_timeline_rebuild", start_ts, "SUCCESS", [bronze_table], job_run_id, end_ts=datetime.now(timezone.utc), rows_read=bronze.count(), rows_written=valid_count, rows_quarantined=error_count)
except Exception as exc:
    upsert_audit(catalog, pipeline_run_id, pipeline_name, "full_timeline_rebuild", start_ts, "FAILED", [bronze_table], job_run_id, end_ts=datetime.now(timezone.utc), error_message=str(exc)[:4000])
    raise